# LEWIS HAMILTON VS MAX VERSTAPPEN (2021)

![skysports-max-verstappen-lewis-hamilton_6480821.jpg](./skysports-max-verstappen-lewis-hamilton_6480821.jpg "skysports-max-verstappen-lewis-hamilton_6480821.jpg")

In [0]:
pip install fastf1

In [0]:
import os
import fastf1
import pandas as pd

os.makedirs("cache", exist_ok=True)
fastf1.Cache.enable_cache('cache')


In [0]:
temporada = fastf1.get_session(2021, 'Abu dhabi', 'R')

## COLETA DOS DADOS GERAIS

In [0]:
GPs = [
    "Bahrain", "Emilia Romagna", "Portugal", "Spain", "Monaco", "Azerbaijan", "France", "Styria","Austria" ,"Grain Britain", "Hungary", "Belgium", "Netherlands",
    "Italy", "Russia", "Turkey", "United States", "Mexico", "Brazil", "Qatar", "Saudi Arabia", "Abu dhabi"
]

In [0]:
dados = []

for corrida in GPs:
    temporada = fastf1.get_session(2021, corrida, "R")
    temporada.load(telemetry=False)

    results = temporada.results [temporada.results['Abbreviation'].isin(['HAM','VER'])
    ][['DriverNumber','Abbreviation','Position','Points','Status']]
    results['race'] = corrida
    results['season'] = 2021
    
    dados.append(results)
    display(results)




In [0]:
spark_df = spark.createDataFrame(results)
display(spark_df)

rivalidade = ['HAM', 'VER']

In [0]:
df_corridas = pd.concat(dados, ignore_index=True)

spark_df_corridas = spark.createDataFrame(df_corridas)
spark_df_corridas.write.mode("overwrite").saveAsTable("temporada_2021")

## CORRIDAS GANHAS

In [0]:
%sql
SELECT 
SUM(CASE WHEN Abbreviation = 'HAM' AND Position = 1 THEN 1 ELSE 0 END) AS HAM_wins,
SUM(CASE WHEN Abbreviation = 'VER' AND Position = 1 THEN 1 ELSE 0 END) AS VER_wins
FROM temporada_2021

## Número DNF

In [0]:
%sql
SELECT
race,
Abbreviation,
Status
FROM temporada_2021
WHERE Status != 'Finished'
AND Abbreviation IN ('HAM','VER')
ORDER BY race

## VOLTAS MAIS RÁPIDAS

In [0]:
dados = []
fastf1.Cache.enable_cache('cache')


for corrida in GPs:
    temporada = fastf1.get_session(2021, corrida, "R")
    temporada.load()

    voltas = temporada.laps.pick_quicklaps()
    volta_rapida = voltas.pick_fastest()

    if volta_rapida is None:
        print(f"Sem volta válida em: {corrida}")
        continue

    row = {
        "season": 2021,
        "race": corrida,
        "driver": volta_rapida['Driver'],
        "lap_time": volta_rapida['LapTime']
    }

    dados.append(row)
df_volta_mais_rapida = pd.DataFrame(dados)
display(df_volta_mais_rapida)

## Criação da tabela de voltas mais rápidas

In [0]:

spark_volta_mais_rapida = spark.createDataFrame(df_volta_mais_rapida)
spark_volta_mais_rapida.write.mode("overwrite").saveAsTable("volta_mais_rapida_2021")

# Consulta SQL

In [0]:
%sql
SELECT driver, count(driver) AS total_voltas_rapidas
FROM volta_mais_rapida_2021
WHERE driver = 'VER' OR driver = 'HAM'
GROUP BY driver



## TOTAL VOLTAS LIDERADAS

In [0]:
dados = []

fastf1.Cache.enable_cache('cache')

for corrida in GPs:

    temporada = fastf1.get_session(2021, corrida, "R")
    temporada.load()

    voltas = temporada.laps

    voltas_lideradas = voltas[voltas['Position'] == 1]

    liderancas = voltas_lideradas.groupby("Driver")['LapNumber'].count().reset_index()

    liderancas['race'] = corrida
    liderancas['season'] = 2021

    dados.append(liderancas)

df_voltas_lideradas = pd.concat(dados, ignore_index=True)

display(df_voltas_lideradas)


# Consulta SQL

In [0]:
%sql
SELECT 
Driver,
SUM(LapNumber) AS total_voltas_lideradas
FROM voltas_lideradas_2021
WHERE Driver = 'HAM' OR Driver = 'VER'
GROUP BY Driver
ORDER BY total_voltas_lideradas DESC